# Figures for the final model
Makes all figures for the final model **v13** on the 58 gold studies.

**Inputs to add:** both `cache384` notebooks (part0 and part1) and the `knee-v10-weights` dataset (with `v13_384_best.pt`).
The competition data is attached automatically when this notebook is created from the competition page.

**Settings:** Accelerator GPU T4. Internet not needed. Then **Run All** (about 10 minutes).

## 1. Setup

In [ ]:
import os, glob, shutil, textwrap, numpy as np, pandas as pd, torch, torch.nn as nn, timm
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import roc_auc_score, roc_curve

WEIGHTS, CACHE_NAME = "v13_384_best.pt", "cache384"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SIZES = [18, 14, 12, 8, 12]; BND = np.cumsum([0] + SIZES)
CENT = [(c, s) for s in range(5) for c in range(BND[s] + 1, BND[s + 1] - 1)]
SLOT = ["Sagittal (fluid-sens.)", "Sagittal (other)", "Coronal (fluid-sens.)", "Coronal (other)", "Axial"]
SLOT_SHORT = ["Sag. fluid", "Sag. other", "Cor. fluid", "Cor. other", "Axial"]
PLANE_COL = ["#1f77b4", "#1f77b4", "#2ca02c", "#2ca02c", "#ff7f0e"]      # sagittal blue, coronal green, axial orange
CASE_COL = {"Correct": "#2ca02c", "Missed": "#d62728", "False alarm": "#ff7f0e"}
FIG_FINDINGS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Effusion", "Fracture"]
OUT = "/kaggle/working/figures"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def find(name, maxdepth=4):
    """Find a file or folder by name anywhere under /kaggle/input."""
    hits = []
    def walk(d, depth):
        try: ents = os.listdir(d)
        except Exception: return
        if name in ents: hits.append(os.path.join(d, name))
        if depth < maxdepth:
            for e in ents:
                p = os.path.join(d, e)
                if e not in ("train_series", "test_series", name) and os.path.isdir(p): walk(p, depth + 1)
    walk("/kaggle/input", 0); return sorted(hits)

def save(fig, name):
    fig.savefig(f"{OUT}/{name}", dpi=220, bbox_inches="tight", facecolor="white")
    plt.close(fig); print("saved", name)

## 2. Load the 58 gold studies from the cache

In [ ]:
R = os.path.dirname(find("train_series.csv")[0])
tr = pd.read_csv(f"{R}/train.csv", dtype={"StudyInstanceUID": str})
gold = tr[tr[LAB].notna().all(1)].reset_index(drop=True)
YG = (gold[LAB].values.astype(float) > 0.5).astype(int)
need = set(gold.StudyInstanceUID); V = {}
for s in sorted(s for d in find(CACHE_NAME) for s in glob.glob(d + "/shard_*.npz")):
    z = np.load(s); zi = z["ids"].astype(str)
    hit = [i for i, sid in enumerate(zi) if sid in need]
    if hit:
        v = z["vol"]
        for i in hit: V[zi[i]] = v[i].copy()
        del v
    if len(V) == len(need): break
print(f"gold studies: {len(gold)} | found in cache: {len(V)}")

## 3. Run the final model (also returns attention)

In [ ]:
ck = torch.load(find(WEIGHTS)[0], map_location="cpu")
class Net(nn.Module):
    def __init__(s):
        super().__init__()
        s.bb = timm.create_model(ck["backbone"], pretrained=False, num_classes=0); Fd = s.bb.num_features
        s.slot = nn.Embedding(5, Fd); s.norm = nn.LayerNorm(Fd)
        s.att = nn.Sequential(nn.Linear(Fd, 256), nn.Tanh(), nn.Dropout(0.2), nn.Linear(256, 12))
        s.W = nn.Parameter(torch.zeros(12, Fd)); s.b = nn.Parameter(torch.zeros(12))
    def forward(s, x, slots):
        B, Kk = x.shape[:2]
        f = torch.cat([s.bb(c) for c in x.flatten(0, 1).split(32)])
        h = s.norm(f.view(B, Kk, -1) + s.slot(slots))
        a = s.att(h).float().softmax(1)                       # attention: (B, windows, 12)
        return (torch.einsum("bkn,bkf->bnf", a, h.float()) * s.W).sum(-1) + s.b, a

dev = "cuda" if torch.cuda.is_available() else "cpu"
model = Net(); model.load_state_dict(ck["model"]); model = model.to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 1, 3, 1, 1)

P, ATT, WIN = [], {}, {}
for sid in gold.StudyInstanceUID:
    vol = V[sid]
    sm = [vol[a:b, ::8, ::8].any() for a, b in zip(BND[:-1], BND[1:])]
    val = [(c, s) for c, s in CENT if sm[s]] or CENT
    x = torch.from_numpy(np.stack([vol[c - 1:c + 2] for c, _ in val]))[None].to(dev).float().div(255)
    sl = torch.tensor([[s for _, s in val]], device=dev)
    with torch.no_grad(), torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
        logit, a = model((x - MEAN) / STD, sl)
    P.append(torch.sigmoid(logit.float())[0].cpu().numpy()); ATT[sid] = a[0].cpu().numpy(); WIN[sid] = val
P = np.array(P)
auc = np.array([roc_auc_score(YG[:, j], P[:, j]) for j in range(12)])
print(f"sanity check - gold macro AUC: {auc.mean():.4f} (training log said 0.8632)")

## 4. Figure 1: the five input views

In [ ]:
ex = gold.StudyInstanceUID[int(YG.sum(1).argmax())]
fig, ax = plt.subplots(1, 5, figsize=(17, 3.9))
for k in range(5):
    a, b = BND[k], BND[k + 1]
    ax[k].imshow(V[ex][(a + b) // 2], cmap="gray")
    ax[k].set_title(SLOT[k], color=PLANE_COL[k], fontweight="bold")
    ax[k].set_xticks([]); ax[k].set_yticks([])
    for sp in ax[k].spines.values(): sp.set_visible(True); sp.set_color(PLANE_COL[k]); sp.set_linewidth(3)
save(fig, "fig1_inputs.png")

## 5. Figure 2: AUC per finding with 95% bootstrap CI

In [ ]:
rng = np.random.default_rng(0); boots = []
for _ in range(1000):
    i = rng.integers(0, len(YG), len(YG))
    if any(YG[i, j].min() == YG[i, j].max() for j in range(12)): continue
    boots.append([roc_auc_score(YG[i, j], P[i, j]) for j in range(12)])
lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
o = np.argsort(auc)
cols = ["#2ca02c" if v >= 0.9 else "#1f77b4" if v >= 0.8 else "#ff7f0e" for v in auc[o]]
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.barh(np.array(LAB)[o], auc[o], color=cols, edgecolor="black", linewidth=0.6,
        xerr=[np.clip(auc[o] - lo[o], 0, None), np.clip(hi[o] - auc[o], 0, None)],   # clip: no negative bars
        error_kw=dict(ecolor="#333333", capsize=3, lw=1))
for y_, (v_, j_) in enumerate(zip(auc[o], o)):
    ax.text(0.505, y_, f"{v_:.2f}   (n+ = {YG[:, j_].sum()})", va="center", color="white",
            fontsize=9.5, fontweight="bold")
ax.axvline(auc.mean(), color="#9467bd", ls="--", lw=1.6)
ax.text(auc.mean() + 0.005, len(LAB) - 0.4, f"macro {auc.mean():.3f}", color="#9467bd", fontweight="bold")
ax.set_xlim(0.5, 1.0); ax.set_xlabel("ROC AUC on 58 gold studies (95% bootstrap CI)")
ax.set_title("Final model (v13): performance per finding")
ax.legend(handles=[Patch(color="#2ca02c", label="AUC >= 0.90"), Patch(color="#1f77b4", label="0.80 to 0.90"),
                   Patch(color="#ff7f0e", label="< 0.80")],
          loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3, frameon=False)
save(fig, "fig2_auc_per_finding.png")

## 6. Figure 2b: ROC curves for all 12 findings

In [ ]:
cmap = plt.get_cmap("tab20")
fig, axs = plt.subplots(3, 4, figsize=(15, 11), sharex=True, sharey=True)
for j, ax in enumerate(axs.flat):
    fpr, tpr, _ = roc_curve(YG[:, j], P[:, j]); col = cmap(j * 2 % 20)
    ax.plot(fpr, tpr, color=col, lw=2.4); ax.fill_between(fpr, tpr, alpha=0.12, color=col)
    ax.plot([0, 1], [0, 1], color="grey", ls=":", lw=1)
    ax.set_title(f"{LAB[j]}  (AUC {auc[j]:.2f}, n+ = {YG[:, j].sum()})", fontsize=11)
    ax.set_aspect("equal")
for ax in axs[-1]: ax.set_xlabel("False positive rate")
for ax in axs[:, 0]: ax.set_ylabel("True positive rate")
fig.suptitle("ROC curves of the final model on the 58 gold studies", fontsize=14, y=1.0)
fig.tight_layout(); save(fig, "fig2b_roc_curves.png")

## 7. Figure 3: correct case, missed case and false alarm per finding

In [ ]:
def top_slices(sid, j, k=3):
    """The k windows with the highest attention weight for finding j."""
    w = ATT[sid][:, j]; idx = np.argsort(-w)[:k]
    return [(WIN[sid][i][0], WIN[sid][i][1], w[i]) for i in idx]

for f in FIG_FINDINGS:
    j = LAB.index(f); pos, neg = np.where(YG[:, j] == 1)[0], np.where(YG[:, j] == 0)[0]
    cases = [("Correct", pos[P[pos, j].argmax()]), ("Missed", pos[P[pos, j].argmin()]),
             ("False alarm", neg[P[neg, j].argmax()])]
    fig, ax = plt.subplots(3, 3, figsize=(11, 11.5))
    for r, (name, i) in enumerate(cases):
        sid, c_col = gold.StudyInstanceUID[i], CASE_COL[name]
        for c_, (c, s, w) in enumerate(top_slices(sid, j)):
            a_ = ax[r, c_]; a_.imshow(V[sid][c], cmap="gray"); a_.set_xticks([]); a_.set_yticks([])
            for sp in a_.spines.values(): sp.set_visible(True); sp.set_color(c_col); sp.set_linewidth(4)
            a_.set_title(f"{SLOT[s]}\nattention weight {w:.2f}", fontsize=9.5, color=PLANE_COL[s])
        gold_txt = "present" if YG[i, j] else "absent"
        ax[r, 0].set_ylabel(f"{name}\np = {P[i, j]:.2f}, gold: {gold_txt}", color=c_col,
                            fontsize=12, fontweight="bold", labelpad=10)
    fig.suptitle(f"{f}: slices the model weighted most", fontsize=14, y=0.995)
    fig.legend(handles=[Patch(facecolor="white", edgecolor=v, linewidth=3, label=k) for k, v in CASE_COL.items()],
               loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.tight_layout(rect=(0, 0.02, 1, 1)); save(fig, f"fig3_{f.replace(' ', '_')}.png")

## 8. Figure 4: which view the model relies on, per finding

In [ ]:
A = np.zeros((12, 5))
for sid in gold.StudyInstanceUID:
    slots = np.array([s for _, s in WIN[sid]])
    for k in range(5): A[:, k] += ATT[sid][slots == k].sum(0)     # attention share per view
A /= len(gold)
fig, ax = plt.subplots(figsize=(8, 7.5))
im = ax.imshow(A, cmap="YlGnBu", vmin=0, vmax=A.max())
ax.set_xticks(range(5)); ax.set_xticklabels(SLOT_SHORT, rotation=25, ha="right")
ax.set_yticks(range(12)); ax.set_yticklabels(LAB)
for t, c in zip(ax.get_xticklabels(), PLANE_COL): t.set_color(c); t.set_fontweight("bold")
for r in range(12):
    for k in range(5):
        ax.text(k, r, f"{A[r, k]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if A[r, k] > 0.6 * A.max() else "black")
fig.colorbar(im, ax=ax, fraction=0.04, label="Mean share of attention")
ax.set_title("Where the model looks: attention per view and finding")
for sp in ax.spines.values(): sp.set_visible(False)
save(fig, "fig4_attention_by_view.png")

## 9. Failure cases (paste this output back for the report)

In [ ]:
err = []
for i in range(len(gold)):
    for j in range(12):
        if abs(P[i, j] - YG[i, j]) > 0.5:
            err.append({"study": gold.StudyInstanceUID[i][-10:], "finding": LAB[j], "gold": YG[i, j],
                        "prob": round(float(P[i, j]), 3), "type": "missed" if YG[i, j] else "false alarm",
                        "report_excerpt": " ".join(str(gold.Report[i]).split())[:300]})
err = pd.DataFrame(err).assign(conf=lambda d: (d.prob - d.gold).abs()).sort_values("conf", ascending=False)
err.drop(columns="conf").head(20).to_csv(f"{OUT}/failure_cases.csv", index=False)
print(f"mistakes with error > 0.5: {len(err)} | by type: {err.type.value_counts().to_dict()}")
print("by finding:", err.finding.value_counts().to_dict())
print("\nattention share per view (rows = findings):")
print(pd.DataFrame(A, index=LAB, columns=SLOT_SHORT).round(2))
for _, r in err.head(8).iterrows():
    print(f"\n[{r.finding} | {r.type} | p={r.prob} gold={r.gold}]\n  " + "\n  ".join(textwrap.wrap(r.report_excerpt, 110)))

## 10. Zip all figures for download

In [ ]:
shutil.make_archive("/kaggle/working/figures", "zip", OUT)
print("saved:", sorted(os.listdir(OUT)))